In [1]:
import torch
import numpy as np

In [2]:
fl_w = torch.tensor([0.1,0.2,0.])
fr_w = torch.tensor([0.1,-0.2,0.])
rl_w = torch.tensor([-0.1,0.2,0.])
rr_w = torch.tensor([-0.1,-0.2,0.])

trk_w = torch.tensor([0,0.0,0.3])

m = 13.10
g = 9.81 

wd = torch.tensor([0,0,m*g,0,0,0])

def skew(v):
    return torch.tensor([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

Jb_t = torch.hstack((torch.cat(((torch.eye(3,3),skew(fl_w - trk_w)))),torch.cat(((torch.eye(3,3),skew(fr_w - trk_w)))),torch.cat(((torch.eye(3,3),skew(rl_w - trk_w)))),torch.cat(((torch.eye(3,3),skew(rr_w - trk_w))))))

f = torch.linalg.pinv(Jb_t) @ wd

In [3]:
f.reshape(4,3)

tensor([[-1.4362e-06,  4.4068e-07,  3.2128e+01],
        [ 1.4362e-06, -9.9556e-07,  3.2128e+01],
        [-1.4362e-06, -3.3893e-06,  3.2128e+01],
        [ 1.4362e-06, -3.3893e-06,  3.2128e+01]])

In [2]:
# Example for n = 3 batch size
fl_w = torch.tensor([[0.1, 0.2, 0.], [0.1, 0.2, 0.], [0.1, 0.2, 0.]])
fr_w = torch.tensor([[0.1, -0.2, 0.], [0.1, -0.2, 0.], [0.1, -0.2, 0.]])
rl_w = torch.tensor([[-0.1, 0.2, 0.], [-0.1, 0.2, 0.], [-0.1, 0.2, 0.]])
rr_w = torch.tensor([[-0.1, -0.2, 0.], [-0.1, -0.2, 0.], [-0.1, -0.2, 0.]])

trk_w = torch.tensor([[0, 0.0, 0.5], [0, 0.0, 0.5], [0, 0.0, 0.5]])

stance_mask = torch.tensor([[1, 1, 0, 1], [1, 1, 0, 0], [1, 0, 1, 0]])
batch_size = fl_w.shape[0]

m = 13
g = 9.81

# Gravity wrench (expanded to batch size)
wd = torch.tensor([0, 0, m * g, 0, 0, 0]).repeat(batch_size, 1, 1)

# Updated skew function to handle batches


def skew(v):
    batch_size = v.shape[0]
    result = torch.zeros((batch_size, 3, 3))
    result[:, 0, 1] = -v[:, 2]
    result[:, 0, 2] = v[:, 1]
    result[:, 1, 0] = v[:, 2]
    result[:, 1, 2] = -v[:, 0]
    result[:, 2, 0] = -v[:, 1]
    result[:, 2, 1] = v[:, 0]
    return result

# Computing Jb_t for batch of vectors


# Eye matrix must be repeated for the batch size
eye_batch = torch.eye(3).repeat(batch_size, 1, 1)

# Compute the skew matrices for each leg
fl_skew = skew(fl_w[:, :3] - trk_w[:, :3])
fr_skew = skew(fr_w[:, :3] - trk_w[:, :3])
rl_skew = skew(rl_w[:, :3] - trk_w[:, :3])
rr_skew = skew(rr_w[:, :3] - trk_w[:, :3])

In [3]:
fl_block = torch.cat((eye_batch, fl_skew), dim=1)
fr_block = torch.cat((eye_batch, fr_skew), dim=1)
rl_block = torch.cat((eye_batch, rl_skew), dim=1)
rr_block = torch.cat((eye_batch, rr_skew), dim=1)

In [4]:
Jb_t = torch.cat((fl_block, fr_block, rl_block, rr_block), dim=2)  # n x 6 x 12


In [5]:
Jb_t.shape

torch.Size([3, 6, 12])

In [6]:
stance_mask_expanded = stance_mask.unsqueeze(-1).expand(batch_size, 4, 3)
stance_mask_for_jb_t = stance_mask_expanded.reshape(batch_size, 1, 12)
stance_mask_for_jb_t = stance_mask_for_jb_t.expand(batch_size, 6, 12)


In [7]:
stance_mask_for_jb_t.shape

torch.Size([3, 6, 12])

In [8]:
Jb_t_masked = Jb_t * stance_mask_for_jb_t

In [10]:
f = torch.linalg.pinv(Jb_t_masked) @ wd.squeeze(1).unsqueeze(2)
f = f.squeeze(-1)

In [11]:
f = f.view(batch_size, 4, 3)

In [13]:
# tensor([[-2.8505e-06,  9.2444e-06,  3.1882e+01],
#         [ 9.5017e-07,  1.1679e-06,  3.1883e+01],
#         [-2.8505e-06, -2.5735e-07,  3.1882e+01],
#         [ 4.7509e-06, -2.5735e-07,  3.1883e+01]])

In [12]:
f

tensor([[[-2.3754e-07, -2.2002e-05,  6.3765e+01],
         [-1.6034e-06,  8.9673e-06,  2.6605e-05],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 9.5017e-07,  7.3193e-06,  6.3765e+01]],

        [[-2.5304e+00,  2.1328e-08,  6.3259e+01],
         [-2.5304e+00,  2.1330e-08,  6.3259e+01],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[-1.3148e-06, -4.9429e+00,  6.1788e+01],
         [-1.0755e-05,  0.0000e+00,  0.0000e+00],
         [-1.3147e-06, -4.9432e+00,  6.1788e+01],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00]]])

In [14]:
f.shape

torch.Size([3, 4, 3])

In [15]:
fl_jac = torch.tensor([[[1.2250e-04, -2.7932e-01, -1.3132e-01],
                        [2.6876e-01, 1.5324e-03, 1.8452e-02],
                        [1.1029e-01, -1.4444e-02, -1.6669e-01],
                        [1.0000e+00, -2.4641e-04, -2.4640e-04],
                        [5.0880e-04, 9.9391e-01, 9.9391e-01],
                        [-2.3505e-03, 1.1022e-01, 1.1022e-01]]])
fr_jac = torch.tensor([[[-3.9831e-04, -2.7986e-01, -1.3166e-01],
                        [2.6894e-01, -1.8533e-03, -1.9103e-02],
                        [-1.1124e-01, -1.4320e-02, -1.6634e-01],
                        [1.0000e+00, -7.7210e-04, -7.7206e-04],
                        [5.0879e-04, 9.9354e-01, 9.9354e-01],
                        [-2.3505e-03, -1.1349e-01, -1.1349e-01]]])
rl_jac = torch.tensor([[[1.2124e-04, -2.8004e-01, -1.6640e-01],
                        [2.6979e-01, -5.1988e-03, 1.4411e-02],
                        [1.0998e-01, 4.6903e-02, -1.3218e-01],
                        [1.0000e+00, -2.5019e-04, -2.5018e-04],
                        [5.0878e-04, 9.9408e-01, 9.9408e-01],
                        [-2.3505e-03, 1.0869e-01, 1.0869e-01]]])
rr_jac = torch.tensor([[[-3.9714e-04, -2.8042e-01, -1.6658e-01],
                        [2.6997e-01, 5.0140e-03, -1.4798e-02],
                        [-1.1052e-01, 4.7025e-02, -1.3191e-01],
                        [1.0000e+00, -7.6518e-04, -7.6514e-04],
                        [5.0882e-04, 9.9387e-01, 9.9387e-01],
                        [-2.3505e-03, -1.1053e-01, -1.1053e-01]]])

In [16]:
fl_tau = -(fl_jac[:, 0:3].transpose(1, 2) @ f[:, 0].unsqueeze(-1)).squeeze(-1)
fr_tau = -(fr_jac[:, 0:3].transpose(1, 2) @ f[:, 1].unsqueeze(-1)).squeeze(-1)
rl_tau = -(rl_jac[:, 0:3].transpose(1, 2) @ f[:, 2].unsqueeze(-1)).squeeze(-1)
rr_tau = -(rr_jac[:, 0:3].transpose(1, 2) @ f[:, 3].unsqueeze(-1)).squeeze(-1)


In [17]:
torch.cat((fl_tau, fr_tau, rl_tau, rr_tau), dim=1)

tensor([[-7.0326e+00,  9.2102e-01,  1.0629e+01,  5.4723e-07, -5.1132e-08,
          4.3856e-06, -0.0000e+00, -0.0000e+00, -0.0000e+00,  7.0473e+00,
         -2.9985e+00,  8.4112e+00],
        [-6.9765e+00,  2.0693e-01,  1.0212e+01,  7.0359e+00,  1.9772e-01,
          1.0189e+01, -0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,
         -0.0000e+00, -0.0000e+00],
        [-5.4861e+00,  9.0004e-01,  1.0391e+01, -4.2838e-09, -3.0099e-06,
         -1.4160e-06, -5.4618e+00, -2.9237e+00,  8.2383e+00, -0.0000e+00,
         -0.0000e+00, -0.0000e+00]])